# 2-Dataset

主线 Dataset 在 `../dataset/lm_dataset.py`。本本用官方 tokenizer + `toydata/` 看五种格式。正式数据文件名是 `pretrain_t2t(_mini).jsonl`、`sft_t2t(_mini).jsonl`、`dpo.jsonl`、`rlaif.jsonl`、`agent_rl.jsonl`。

和 MiniMind2 的关键差别：

- tokenizer 目录就是 `../model/`，不再是 `model/minimind_tokenizer`
- 特殊标记是 `<|im_start|>` / `<|im_end|>`，不是 `<s>` / `</s>`
- SFT 返回 `(input_ids, labels)`，pad 位置 label=`-100`，不再手传 `loss_mask`
- SFT 已混 Tool Call / 思考字段；另有 `RLAIFDataset`、`AgentRLDataset`


In [ ]:
import os, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
print("cwd:", os.getcwd())

from transformers import AutoTokenizer
from dataset.lm_dataset import (
    PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset,
    pre_processing_chat, post_processing_chat,
)
tokenizer = AutoTokenizer.from_pretrained("../model")
print(tokenizer.vocab_size, tokenizer.bos_token, tokenizer.eos_token)


## Pretrain：`{"text": ...}`

目标是 next-token prediction。Dataset 会包上 bos/eos，pad 到 `max_length`，pad 的 label 为 `-100`。


In [ ]:
import json
with open("./toydata/pretrain_data.jsonl") as f:
    print(json.loads(next(f))["text"][:80], "...")
ds = PretrainDataset("./toydata/pretrain_data.jsonl", tokenizer, max_length=64)
input_ids, labels = ds[0]
print(input_ids.shape, labels.shape, "num_ignored", int((labels == -100).sum()))
print(tokenizer.decode(input_ids[labels != -100][:20]))


## SFT：conversations + tools / tool_calls / reasoning_content

`create_chat_prompt` 走官方 `chat_template`。loss 只打在 assistant 段（含 tool call）。`pre_processing_chat` 会按概率补 system；`post_processing_chat` 会按概率丢掉空 `<think>` 标签。


In [ ]:
with open("./toydata/sft_data.jsonl") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        roles = [m["role"] for m in obj["conversations"]]
        extra = [k for k in obj["conversations"][0] if k not in ("role", "content")]
        print(i, roles, extra)
ds = SFTDataset("./toydata/sft_data.jsonl", tokenizer, max_length=128)
input_ids, labels = ds[0]
print("supervised tokens", int((labels != -100).sum()), "/", len(labels))
print(tokenizer.decode(input_ids, skip_special_tokens=False)[:400])


Tool Call 样本会被模板展开成 `<tool_call>...</tool_call>` / `<tool_response>...</tool_response>`：


In [ ]:
sample = json.loads(open("./toydata/sft_data.jsonl").read().splitlines()[-1])
prompt = ds.create_chat_prompt(sample["conversations"])
print(prompt)


## DPO：chosen / rejected

一对偏好回复，loss 只作用在 assistant token 上。


In [ ]:
ds = DPODataset("./toydata/dpo_data.jsonl", tokenizer, max_length=128)
batch = ds[0]
print({k: v.shape for k, v in batch.items()})


## RLAIF：只返回 prompt，留给 rollout 续写

最后一个 assistant 不进入 prompt，`add_generation_prompt=True`，并按 `thinking_ratio` 决定是否打开 `open_thinking`。


In [ ]:
ds = RLAIFDataset("./toydata/rlaif_data.jsonl", tokenizer, max_length=128, thinking_ratio=1.0)
print(ds[0]["prompt"])


## Agent RL：messages + tools + gt

多轮 Tool-Use。`gt` 是可校验的最终答案（例如数学结果），给 `train_agent.py` 做规则奖励。


In [ ]:
ds = AgentRLDataset("./toydata/agent_data.jsonl", tokenizer)
item = ds[0]
print(item.keys())
print("tools:", item["tools"])
print("gt:", item["gt"])
print("last user:", item["messages"][-1])
